## Init

In [ ]:
# ---------------- Imports ----------------
import os

from datetime import datetime
from transformers import AutoTokenizer

import torch
import yaml

from sentence_transformers import SentenceTransformer

from elicitation.metrics.progression import progression
from elicitation.metrics.turn_length_ratio import turn_length_ratio
from elicitation.metrics.utils import load_dialogues


In [ ]:
# ---------------- Args ----------------
# Paths
embedding_model_choice_name = "sentence-transformers/all-MiniLM-L12-v2"
#cross_encoder_model_choice_name = "cross-encoder/stsb-roberta-large"
tokenizer_model_choice_name = "meta-llama/Llama-3.2-3B-Instruct"

dataset_choice_name = "evaluation/generated-utterances-dialogue/20251227T2034-20251227t0851-deepseek-r1-distill-llama-8b-seq-std-m3trained/generated"





# Constants
PROGRESSION_K = 5
PROGRESSION_GAMMA = 0.5
#CONVERSATIONAL_CONTROL_K = 2 
#CONVERSATIONAL_CONTROL_GAMMA = 0.9



In [ ]:
# ---------------- Config ----------------
timestamp = datetime.now().strftime("%Y_%m_%d_%H_%M_%S")

with open("../../config/config.yaml", "r") as f:
    config = yaml.safe_load(f)

proj_store = config["paths"]["proj_store"]

data_path = os.path.join(config["paths"]["proj_store"], "data")

dataset_path = os.path.join(proj_store, dataset_choice_name)

models_folderpath = config["paths"]["models"]

embedding_model_choice = os.path.join(models_folderpath, embedding_model_choice_name)
#cross_encoder_model_choice = os.path.join(models_folderpath, cross_encoder_model_choice_name)
tokenizer_model_choice = os.path.join(models_folderpath, tokenizer_model_choice_name)


save_folder_path = os.path.join(proj_store, "evaluation", "interaction-metrics", f"{dataset_choice_name}")
os.makedirs(save_folder_path, exist_ok=True)


# Load llama tokenizer
tokenizer_model = AutoTokenizer.from_pretrained(tokenizer_model_choice, trust_remote_code=True)

# Load sentence embedding model
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)
embedding_model = SentenceTransformer(embedding_model_choice, device=device)






In [ ]:
all_dialogues = list(load_dialogues(dataset_path))

print("Loaded", len(all_dialogues), "dialogues")



## Progression

In [ ]:
progression_df = progression(
    dialogues=all_dialogues, 
    embedding_model=embedding_model, 
    device=device, 
    k=PROGRESSION_K, 
    gamma=PROGRESSION_GAMMA, 
    group_by="domain", 
    sort_by="domain"
)

display(progression_df)



In [ ]:
# Save to CSV
progression_df.to_csv(os.path.join(save_folder_path, f"progression.csv"), index=False)



## Turn-Length Ratio

In [ ]:
turn_length_df = turn_length_ratio(
    dialogues=all_dialogues, 
    tokenizer_model=tokenizer_model, 
    group_by="domain", 
    sort_by="domain"
)

display(turn_length_df)



In [ ]:
# Save to CSV
turn_length_df.to_csv(os.path.join(save_folder_path, f"turn_length_ratio.csv"), index=False)

